# Composition vs Inheritance

**OOP-in-Python — Advanced**  
IS-A vs HAS-A relationships, dependency injection, mixins, and cooperative inheritance.

## The Core Idea

| Principle | Relationship | Keyword | Example |
|---|---|---|---|
| **Inheritance** | IS-A | `extends` | `Car` IS-A `Vehicle` |
| **Composition** | HAS-A | `contains` | `Car` HAS-A `Engine` |

**Favor composition over inheritance** — *Gang of Four*

## 1. Inheritance — IS-A

A **child class** inherits from a **parent class** — code reuse + extension.

In [ ]:
# IS-A relationship: Penguin IS-A Bird
class Bird:
    def __init__(self, name: str):
        self.name = name

    def eat(self) -> str:
        return f"{self.name} is eating"

class Penguin(Bird):  # IS-A
    def swim(self) -> str:
        return f"{self.name} is swimming"

p = Penguin("Emperor")
print(p.eat())   # Inherited
print(p.swim())  # Specialized

## 2. Composition — HAS-A

A class **contains** instances of other classes — more flexible, looser coupling.

In [ ]:
# HAS-A relationship: Car HAS-A Engine and Wheels
class Engine:
    def start(self) -> str:
        return "Engine started"

    def stop(self) -> str:
        return "Engine stopped"

class Wheels:
    def rotate(self) -> str:
        return "Wheels rotating"

class Car:
    def __init__(self):
        self.engine = Engine()  # Composition
        self.wheels = [Wheels() for _ in range(4)]  # Composition

    def drive(self) -> str:
        return f"{self.engine.start()} → {self.wheels[0].rotate()}"

c = Car()
print(c.drive())

## 3. When to Prefer Composition

| Scenario | Prefer |
|---|---|
| "Is-a" relationship is genuine | Inheritance |
| "Has-a" or "uses-a" relationship | Composition |
| Need to change behavior at runtime | Composition |
| Deep inheritance hierarchies (>3 levels) | Composition |
| Class is a collection of behaviors | Composition / Mixins |
| Needs to inherit from many sources | Composition (avoid diamond) |

In [ ]:
# Bad: deep inheritance for behavior variation
class Animal:
    pass
class FlyingAnimal(Animal):
    pass
class SwimmingFlyingAnimal(FlyingAnimal):
    pass  # Fragile!

# Better: compose behaviors
class Flyable:
    def fly(self) -> str:
        return "Flying"

class Swimmable:
    def swim(self) -> str:
        return "Swimming"

class Duck:
    def __init__(self):
        self.fly_behavior = Flyable()
        self.swim_behavior = Swimmable()

    def fly(self) -> str:
        return self.fly_behavior.fly()

    def swim(self) -> str:
        return self.swim_behavior.swim()

d = Duck()
print(d.fly(), d.swim())

## 4. Dependency Injection

A form of composition where dependencies are **injected** (passed in) rather than created internally.

In [ ]:
from typing import Protocol

class Logger(Protocol):
    def log(self, message: str): ...

class ConsoleLogger:
    def log(self, message: str):
        print(f"[Console] {message}")

class FileLogger:
    def __init__(self, filename: str):
        self.filename = filename

    def log(self, message: str):
        with open(self.filename, 'a') as f:
            f.write(f"[File] {message}\n")

class UserService:
    # Logger is injected — not created here!
    def __init__(self, logger: Logger):
        self.logger = logger

    def create_user(self, name: str):
        # Business logic...
        self.logger.log(f"User '{name}' created")

# Inject different loggers
svc = UserService(ConsoleLogger())
svc.create_user("Alice")  # [Console] User 'Alice' created

# svc = UserService(FileLogger("users.log"))  # Easy to swap!

## 5. Mixins

Mixins are **small classes** that provide specific, reusable functionality. Used with multiple inheritance.

In [ ]:
class TimestampMixin:
    """Mixin that adds creation timestamp."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        from datetime import datetime
        self.created_at = datetime.now()

class JSONMixin:
    """Mixin that adds JSON serialization."""
    def to_json(self) -> dict:
        return {k: str(v) for k, v in self.__dict__.items() if not k.startswith('_')}

class LoggableMixin:
    """Mixin that adds logging capability."""
    def log(self, msg: str):
        print(f"[{self.__class__.__name__}] {msg}")

class User(TimestampMixin, JSONMixin, LoggableMixin):
    def __init__(self, name: str, email: str):
        super().__init__()  # Calls TimestampMixin.__init__ via MRO
        self.name = name
        self.email = email

u = User("Alice", "alice@example.com")
print(u.created_at)   # From TimestampMixin
print(u.to_json())    # From JSONMixin
u.log("User created") # From LoggableMixin

## 6. Cooperative Inheritance via `super()`

When using mixins or multiple inheritance, always use `super()` consistently for cooperative MRO.

In [ ]:
class A:
    def __init__(self):
        print("A.__init__")

class LogMixin:
    def __init__(self):
        print("LogMixin.__init__")
        super().__init__()  # Cooperative

class ValidateMixin:
    def __init__(self):
        print("ValidateMixin.__init__")
        super().__init__()  # Cooperative

class MyClass(LogMixin, ValidateMixin, A):
    def __init__(self):
        print("MyClass.__init__")
        super().__init__()  # Calls LogMixin → ValidateMixin → A

print(MyClass.__mro__)
m = MyClass()
# MyClass → LogMixin → ValidateMixin → A → object
# All __init__ methods run exactly once!

---
## Problem 1: Build a Computer System Using Composition

**Difficulty: Medium**

Model a Computer using composition: CPU, RAM, Storage, GPU — each as separate classes.

In [ ]:
class CPU:
    def __init__(self, cores: int, speed_ghz: float):
        self.cores = cores
        self.speed = speed_ghz

    def compute(self) -> str:
        return f"Computing on {self.cores} cores @ {self.speed}GHz"

class RAM:
    def __init__(self, size_gb: int):
        self.size_gb = size_gb

    def load(self) -> str:
        return f"Loading data into {self.size_gb}GB RAM"

class Storage:
    def __init__(self, size_gb: int, ssd: bool = True):
        self.size_gb = size_gb
        self.ssd = ssd

    def read(self) -> str:
        drive_type = "SSD" if self.ssd else "HDD"
        return f"Reading from {self.size_gb}GB {drive_type}"

class GPU:
    def __init__(self, model: str, vram_gb: int):
        self.model = model
        self.vram = vram_gb

    def render(self) -> str:
        return f"Rendering on {self.model} ({self.vram}GB VRAM)"

class Computer:
    def __init__(self, cpu: CPU, ram: RAM, storage: Storage, gpu: GPU = None):
        self.cpu = cpu
        self.ram = ram
        self.storage = storage
        self.gpu = gpu  # Optional component (e.g., integrated graphics)

    def boot(self) -> str:
        return f"Booting...\n{self.ram.load()}\n{self.storage.read()}"

    def run_program(self, name: str) -> str:
        result = f"Running {name}:\n{self.cpu.compute()}"
        if self.gpu:
            result += f"\n{self.gpu.render()}"
        return result

# Build a high-end PC
pc = Computer(
    cpu=CPU(8, 4.5),
    ram=RAM(32),
    storage=Storage(1000, ssd=True),
    gpu=GPU("NVIDIA RTX 4080", 16)
)

print(pc.boot())
print("---")
print(pc.run_program("Cyberpunk 2077"))

## Problem 2: Logger Mixin Pattern

**Difficulty: Medium**

Create a reusable mixin that adds logging to any class.

In [ ]:
from datetime import datetime

class LoggerMixin:
    """Mixin that adds timestamped logging to any class."""
    def log(self, level: str, message: str):
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"[{ts}] [{level}] [{self.__class__.__name__}] {message}")

    def info(self, message: str):
        self.log("INFO", message)

    def error(self, message: str):
        self.log("ERROR", message)

    def warn(self, message: str):
        self.log("WARN", message)

class DatabaseConnection(LoggerMixin):
    def connect(self, url: str):
        self.info(f"Connecting to {url}")
        # Simulate connection...
        self.info("Connected successfully")

    def query(self, sql: str):
        self.info(f"Executing: {sql}")
        return ["result1", "result2"]

class PaymentGateway(LoggerMixin):
    def charge(self, amount: float, card: str):
        last4 = card[-4:]
        self.info(f"Charging ${amount} to card ending in {last4}")
        # Simulate...
        self.warn("High amount detected, flagging for review")
        return {"success": True, "txn_id": "TXN123"}

db = DatabaseConnection()
db.connect("postgres://localhost:5432/mydb")
db.query("SELECT * FROM users")

print("---")

pg = PaymentGateway()
pg.charge(999.99, "4111111111111111")

## Key Takeaways

- **Inheritance (IS-A)**: good for genuine taxonomic relationships; can become inflexible.
- **Composition (HAS-A)**: more flexible, looser coupling, runtime-swappable behavior.
- **Dependency Injection**: pass dependencies in rather than creating them internally.
- **Mixins**: reusable behavior chunks via multiple inheritance — use cooperative `super()`.
- **Rule of thumb**: if you're past 3 levels of inheritance, switch to composition.
- **Testing benefit**: composition makes mocking and unit testing far easier.